In [3]:
%cd /mnt/sdd1/atharvas/formulacode/datasmith/
import datetime
import json
from pathlib import Path

import pandas as pd

from datasmith.docker.context import ContextRegistry

curr_date: str = datetime.datetime.now().isoformat()

/mnt/sdd1/atharvas/formulacode/datasmith


In [4]:
# traverse and find all context_registry*.json files under scratch/**
from datasmith.notebooks.utils import merge_registries, update_cr

registries = Path("scratch").rglob("*context_registry*.json")


merged_json = merge_registries(registries)

registry = update_cr(ContextRegistry.deserialize(payload=json.dumps(merged_json)))
len(registry.registry)

3406

In [5]:
registry.save_to_file(Path(f"scratch/merged_context_registry_{curr_date}.json"))

14:33:47 INFO     datasmith.docker.context: Context registry saved to scratch/merged_context_registry_2025-09-09T14:32:37.382974.json


In [6]:
# merge the registries in the scratch/artifacts/processed folders.

registries = Path("scratch/artifacts/processed/downloads/").rglob("**/*context_registry*.json")
merged_json = merge_registries(registries)
registry = update_cr(ContextRegistry.deserialize(payload=json.dumps(merged_json)))
len(registry.registry)

scratch/artifacts/processed/downloads/merged_context_registry_2025-09-09T00:38:42.145419.json : 1016 entries
scratch/artifacts/processed/downloads/merged_context_registry_2025-09-09T01:32:38.179134.json : 1156 entries
scratch/artifacts/processed/downloads/numpy/context_registry.json : 67 entries
scratch/artifacts/processed/downloads/distributed/context_registry.json : 102 entries
scratch/artifacts/processed/downloads/sklearn/context_registry.json : 38 entries
scratch/artifacts/processed/downloads/pandas/context_registry.json : 528 entries
scratch/artifacts/processed/downloads/pandas2/context_registry.json : 43 entries
scratch/artifacts/processed/downloads/scikit-image/context_registry.json : 28 entries
scratch/artifacts/processed/downloads/dask/context_registry.json : 137 entries
scratch/artifacts/processed/downloads/astropy/context_registry.json : 55 entries
scratch/artifacts/processed/downloads/scipy/context_registry.json : 206 entries


1156

In [16]:
t = registry.get("dask/distributed/01b607c804c2023d33ba303db634acfabc938e4e:pkg")

In [24]:
print(t.to_dict())

{'dockerfile_data': '# syntax=docker/dockerfile:1.7\n\nARG BASE_IMAGE=buildpack-deps:jammy\nFROM ${BASE_IMAGE} AS base\n\nRUN apt-get update && \\\n    apt-get install -y --no-install-recommends \\\n        jq cmake ninja-build && \\\n    rm -rf /var/lib/apt/lists/*\n\nRUN curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest \\\n      | tar -xvj -C /usr/local/bin --strip-components=1 bin/micromamba\n\nENV MAMBA_ROOT_PREFIX=/opt/conda \\\n    PATH=/opt/conda/bin:$PATH \\\n    MAMBA_DOCKERFILE_ACTIVATE=1 \\\n    OPENBLAS_NUM_THREADS=1 \\\n    MKL_NUM_THREADS=1 \\\n    OMP_NUM_THREADS=1\n\nRUN micromamba install -y -p $MAMBA_ROOT_PREFIX -c conda-forge \\\n        python=3.10 \\\n        git asv pyperf mamba conda libmambapy && \\\n    micromamba clean --all --yes\n\nRUN mkdir -p /workspace /output\nWORKDIR /workspace\n\n\nCOPY docker_build_base.sh /workspace/docker_build_base.sh\nRUN chmod +x /workspace/docker_build_base.sh && \\\n    /workspace/docker_build_base.sh\n\nRUN micro

In [7]:
registry.save_to_file(Path(f"scratch/artifacts/processed/downloads/merged_context_registry_{curr_date}.json"))

14:34:27 INFO     datasmith.docker.context: Context registry saved to scratch/artifacts/processed/downloads/merged_context_registry_2025-09-09T14:32:37.382974.json


In [12]:
df = pd.read_parquet("scratch/artifacts/pipeflush/commits_perfonly.parquet")
df.head()

,sha,date,message,total_additions,total_deletions,total_files_changed,files_changed,patch,has_asv,file_change_summary,kind,repo_name
0,3263e718a6cc2d10ae4e3e4ba4d4c7ed41ee12e8,2024-07-06T09:38:32+08:00,Merge pull request #125 from Kai-Striega/broad...,133,66,3,numpy_financial/_financial.py\nnumpy_financial...,From a00ab5f0443d2f1c52875b70f19f334c73a17729 ...,True,| File | Lines Added | Lines Removed | Total C...,commit,numpy/numpy-financial
1,3f67c275e1e575c902027ca07586b9d35f38033a,2024-05-07T15:04:23+10:00,Merge pull request #122 from Eugenia-Mazur/irr...,62,47,1,numpy_financial/_financial.py,From a00ab5f0443d2f1c52875b70f19f334c73a17729 ...,True,| File | Lines Added | Lines Removed | Total C...,commit,numpy/numpy-financial
2,5c66fb06ec95192d4b427b4de171b6ab9e1528a6,2024-05-04T11:03:28+10:00,Merge pull request #124 from Kai-Striega/confi...,8,18,3,asv.conf.json\ndoc/source/dev/running_the_benc...,From 646f292a26089dc212e4315f0939c183f660ccea ...,True,| File | Lines Added | Lines Removed | Total C...,commit,numpy/numpy-financial
3,6c40b8efb727eacf8a865789afbe65ee2d4bb5c0,2024-04-04T14:13:19+11:00,Merge pull request #120 from Kai-Striega/enh/n...,6,2,1,numpy_financial/_cfinancial.pyx,From 5b134ac31419fea11db1dda25315d1bd192d8430 ...,True,| File | Lines Added | Lines Removed | Total C...,commit,numpy/numpy-financial
4,858358697fce8fb96530f9c299d285286e5192e5,2024-04-04T10:36:54+11:00,Merge pull request #118 from Kai-Striega/enh/n...,95,29,3,numpy_financial/_cfinancial.pyx\nnumpy_financi...,From 6b6f7b5ba1a50a1199c408b99538c397ef54d0ba ...,True,| File | Lines Added | Lines Removed | Total C...,commit,numpy/numpy-financial


In [ ]:
# scratch/artifacts/processed/synthetic_commits_perfonly.parquet
import pandas as pd

rows = []
for task in registry.registry:
    rows.append({
        "sha": task.sha,
        "repo_name": f"{task.owner}/{task.repo}",
        # date must be in this format:
        # 2024-07-06T09:38:32+08:00
        # convert from linux time to this format:
        "date": datetime.datetime.fromtimestamp(task.commit_date).astimezone().isoformat(),
        "kind": "commit",
        "has_asv": True,
    })

df = pd.DataFrame(rows)
df.to_parquet("scratch/artifacts/processed/synthetic_commits_perfonly.parquet", index=False)

In [ ]:
# import shutil
# import tempfile


# def merge_dict(d1, d2):
#     for key in d2:
#         if key in d1:
#             if isinstance(d1[key], dict) and isinstance(d2[key], dict):
#                 merge_dict(d1[key], d2[key])
#             elif isinstance(d1[key], list) and isinstance(d2[key], list):
#                 d1[key].extend(d2[key])
#             else:
#                 print(f"Cannot merge key {key}, different types or non-mergeable")
#         else:
#             d1[key] = d2[key]
#     return d1


# def merge_results(results, dest: Path):
#     with tempfile.TemporaryDirectory() as tmpdir:
#         for result in results:
#             # a lot of *.pkl files that can be copied over directly.
#             for file in result.rglob("*.pkl"):
#                 shutil.copy(file, tmpdir)
#             # a couple of jsonl files that should be merged.
#             for file in result.rglob("*.jsonl"):
#                 with open(file) as f:
#                     lines = f.readlines()
#                 with open(Path(tmpdir) / file.name, "a") as f:
#                     f.writelines(lines)
#             # a couple of json files that should be merged.
#             for file in result.rglob("*.json"):
#                 with open(file) as f:
#                     data = json.load(f)
#                 # with open(Path(tmpdir) / file.name, "w") as f:
#                 #     json.dump(data, f)
#                 if Path(tmpdir).joinpath(file.name).exists():
#                     with open(Path(tmpdir) / file.name) as f:
#                         existing_data = json.load(f)
#                     merged_data = merge_dict(existing_data, data)
#                     with open(Path(tmpdir) / file.name, "w") as f:
#                         json.dump(merged_data, f)
#                 else:
#                     with open(Path(tmpdir) / file.name, "w") as f:
#                         json.dump(data, f)
#         dest_path = dest / f"results_synthesis_merged_{curr_date}"
#         dest_path.mkdir(parents=True, exist_ok=True)
#         for file in Path(tmpdir).iterdir():
#             shutil.copy(file, dest_path / file.name)


# merge_results(
#     list(Path("scratch/artifacts/processed").rglob("results_synthesis")),
#     Path("scratch/artifacts/processed")
# )